# Step 04. Traffic Manager — 다수 차량 자동 주행

> **참고**: NGV_E2E_Education (Hanyang Univ.) — author Seokhwan Jeong (shjeong00@hanyang.ac.kr)  
> CARLA 0.9.16  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Traffic Manager(TM)로 여러 차량을 autopilot 주행시키고, 속성·경로를 제어합니다.

> ⚠️ **이 노트북의 CARLA 코드 셀은 시뮬레이터가 실행 중이어야 동작합니다.**  
> `./CarlaUE4.sh` 로 서버를 켠 뒤 위에서부터 실행하세요.

## 학습 목표

1. TM 연결과 동기화
2. 다수 차량 spawn + autopilot
3. 차량/전역 TM 속성 (라이트·차선변경·차간거리·속도)
4. 경로 강제 (`set_path` / `set_route`)
5. 일괄 정리

## 1. 환경 + TM 준비

### 이론 배경

Traffic Manager는 autopilot 차량들의 거동(속도·차선·신호 반응)을 총괄하는 모듈입니다.  
동기 모드에서는 `traffic_manager.set_synchronous_mode(True)` 로 World와 동기화합니다.

In [1]:
import carla, random

client = carla.Client('localhost', 2000)
client.set_timeout(10.0)
world = client.get_world()
traffic_manager = client.get_trafficmanager(8000)
tm_port = traffic_manager.get_port()
print('서버 + Traffic Manager 연결 (비동기 모드, port', tm_port, ')')

def destroy_all_vehicles(world, client):
    ids = [v.id for v in world.get_actors().filter('vehicle.*')]
    if ids: client.apply_batch([carla.command.DestroyActor(x) for x in ids])

서버 + Traffic Manager 연결 (비동기 모드, port 8000 )


## 2. 다수 차량 Spawn

### 이론 배경

spawn point마다 무작위 4륜 차량을 `try_spawn_actor` 로 배치하고 `set_autopilot(True, tm_port)` 로 TM에 연결합니다.  
점유된 자리는 `None` 이 반환되므로 건너뜁니다.

In [2]:
NUM_VEHICLES = 30
spawn_points = world.get_map().get_spawn_points()

# 4륜 차량 블루프린트만 사용
vehicle_bps = [bp for bp in world.get_blueprint_library().filter('vehicle.*')
               if bp.has_attribute('number_of_wheels') and int(bp.get_attribute('number_of_wheels')) == 4]

vehicles = []
for sp in spawn_points[:NUM_VEHICLES]:
    bp = random.choice(vehicle_bps)
    if bp.has_attribute('color'):
        bp.set_attribute('color', random.choice(bp.get_attribute('color').recommended_values))
    v = world.try_spawn_actor(bp, sp)     # 점유된 자리면 None
    if v is not None:
        v.set_autopilot(True, tm_port)
        vehicles.append(v)
print(f'{len(vehicles)}대 차량 spawn + autopilot 연결')


30대 차량 spawn + autopilot 연결


## 3. TM 속성 설정

### 이론 배경

| 함수 | 의미 |
|------|------|
| `update_vehicle_lights` | 자동 전조등 |
| `auto_lane_change` | 자동 차선 변경 허용 |
| `distance_to_leading_vehicle` | 앞차와 최소 간격 |
| `vehicle_percentage_speed_difference` | 제한속도 대비 %(음수=빠르게) |

전역은 `global_*` 함수로 모든 차량에 일괄 적용합니다.

In [ ]:
# Traffic Manager 속성 — 차량 단위 / 전역 설정
for v in vehicles:
    traffic_manager.update_vehicle_lights(v, True)            # 자동 라이트
    traffic_manager.auto_lane_change(v, True)                 # 자동 차선 변경 허용
    traffic_manager.distance_to_leading_vehicle(v, 3.0)       # 앞차와 최소 3m
    traffic_manager.vehicle_percentage_speed_difference(v, -10)  # 제한속도 +10% (음수=빠르게)

# 전역 설정
traffic_manager.global_percentage_speed_difference(0)        # 전체 기본 속도
print('TM 속성 적용: 라이트/차선변경/차간거리/속도')


## 4. 경로 강제

### 이론 배경

- `set_path(vehicle, [Location, ...])`: 지정 위치들을 순서대로 통과
- `set_route(vehicle, ['Left','Straight',...])`: 교차로마다 방향 선택

시나리오 실습(Step 05~06)에서 특정 차량을 원하는 경로로 보낼 때 사용합니다.

In [ ]:
# 특정 차량에 경로 강제 — set_path(위치 리스트) 또는 set_route(방향 명령)
target = vehicles[0]
traffic_manager.ignore_lights_percentage(target, 100)        # 신호 무시(실험용)
traffic_manager.auto_lane_change(target, False)

# (a) 위치 기반 경로: spawn point 인덱스들을 거쳐가도록
path_idx = [50, 60, 70, 80]
path_locations = [spawn_points[i % len(spawn_points)].location for i in path_idx]
traffic_manager.set_path(target, path_locations)

# (b) 방향 명령 기반 경로 (교차로에서의 선택): 'Left'/'Right'/'Straight'
# route_cmds = ['Straight', 'Left', 'Straight', 'Right']
# traffic_manager.set_route(target, route_cmds)
print('target 차량에 경로(set_path) 부여')


## 5. 주행 및 정리

### 이론 배경

동기 모드이므로 `world.tick()` 을 반복해야 시뮬레이션이 진행됩니다.  
끝나면 모든 차량을 일괄 제거하고 설정을 복구합니다.

다음: Step 05에서 ego + 선행차로 **판단 시나리오**를 직접 만듭니다.

In [ ]:
# 다수 차량 연속 주행은 동기 tick 루프가 필요 -> 스크립트로 실행
print('터미널에서 실행: python3 scripts/run_tm_attribute.py')
print('  (30대 spawn + autopilot + TM 속성, Ctrl+C 로 종료)')
print('경로 강제 데모: scripts/run_tm_setpath.py , scripts/run_tm_setroute.py')

In [ ]:
destroy_all_vehicles(world, client)
print('전체 차량 제거 완료 (비동기 모드)')